# 08 MiniMind 模型蒸馏

本 Notebook 学习两种蒸馏方法。它们传递 Teacher 信息的层面不同：

```text
Sequence-level Distillation: 传递完整生成文本 → Student SFT
Logit Distillation:          传递每个 token 位置的概率分布 → Student KL loss
```

两种方法可以独立使用。Teacher 和 Student 的 vocabulary 对齐时，也可以先生成 Teacher 文本，再在训练这些文本时加入 Teacher logits。本阶段将它们拆成两条实验链路，分别观察输入、Teacher 信号、loss 和 tokenizer 条件。

In [1]:
# 加载本阶段配置与核心 loss 实现。
from pathlib import Path

import torch

from llm_learning.minimind.distill_config import (
    load_logit_distillation_config,
    load_sequence_teacher_config,
)
from llm_learning.minimind.distill_data import (
    build_gsm8k_split,
    gsm8k_gold_answer,
    verify_math_answer,
)
from llm_learning.minimind.distillation import distillation_losses

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_DIR = ROOT / "docs/stages/08_distillation/configs"
training_config = load_logit_distillation_config(CONFIG_DIR / "ce.json")
print(f"PyTorch: {torch.__version__}")
print(f"training device: {training_config.device}")

PyTorch: 2.9.1
training device: cuda:0


## 1. 两种蒸馏方法传递什么

假设 Teacher 回答问题“12 个苹果平分给 3 人，每人几个？”。

Sequence-level Distillation 保存 Teacher 的完整文本：

```text
12 / 3 = 4，所以每人 4 个。
```

Student 对这段文本执行普通 SFT：Student tokenizer 先将文本编码为 token IDs，再将 assistant response 设为 labels。Teacher 只提供文本，Student 会用自己的 tokenizer 重新编码，所以两者的 tokenizer 可以不同。

Logit Distillation 保留 Teacher 对下一个 token 的整个分布。例如：

| token | Teacher probability |
| --- | ---: |
| `4` | 0.82 |
| `four` | 0.10 |
| `3` | 0.03 |
| 其他 token | 0.05 |

Student 会学习这些相对概率。概率分布的第 $v$ 项对应 vocabulary 中 token ID $v$。Teacher 和 Student 的第 $v$ 项必须表示同一个 token，因此本阶段的 Logit Distillation 共用 MiniMind tokenizer 和 6,400 个 token 的 vocabulary。tokenizer 不一致时，同一句文本会被切成不同 token，两个 logits 位置无法直接比较；此时可以直接使用 Sequence-level Distillation，若要继续做 Logit Distillation 则需另建 vocabulary 或 token span 映射。

这里的三个缩写分别表示：

- CE：Cross-Entropy，用 gold next token 监督 Student；
- KD：Knowledge Distillation，表示知识蒸馏训练；本阶段具体使用 KL divergence 作为 KD loss；
- KL：Kullback–Leibler divergence，本阶段用它比较 Teacher 与 Student 的词表概率分布。

| 项目 | Sequence-level Distillation | Logit Distillation |
| --- | --- | --- |
| Teacher 信号 | 完整 response 文本 | 每个有效位置的词表分布 |
| Student loss | response tokens 的 CE | Teacher–Student KL，可与 CE 组合 |
| tokenizer 条件 | 可以不同 | 需要词表与 token IDs 对齐 |
| 本阶段 Teacher | Qwen3.5-9B GGUF | MiniMind 198M MoE |
| 本阶段 Student | MiniMind 64M Dense | MiniMind 64M Dense |

与前面完成的训练相比：Pretraining 和普通 SFT 都只运行当前模型的 forward 与 backward；Sequence-level 先增加一次 Teacher 文本生成，Student 训练部分仍是 SFT；Logit Distillation 在每个训练 batch 中额外运行 Teacher forward，再对 Student backward。

## 2. Sequence-level Distillation 的数据流

本阶段用本地 `Qwen3.5-9B-Q4_K_M.gguf` 作为 Teacher。LM Studio 负责加载 GGUF 并提供 API。`mmproj` 是多模态模型的视觉投影文件，用于把图像编码器输出转换成语言模型可接收的 embedding；GSM8K 只有文本输入，因此不加载它。

```text
GSM8K question
      ↓
Qwen3.5-9B response
      ↓
Math-Verify 与 gold answer 比较
      ↓ 通过
user / assistant conversations
      ↓
MiniMind response-only SFT
```

Math-Verify 先解析数学表达式，再判断预测结果与 gold answer 是否等价。当前代码优先提取 Teacher response 中 `\boxed{...}` 的 LaTeX 内容，也支持一般数值或表达式。例如 `1/2`、`0.5` 和 `\frac{1}{2}` 可以解析为等价结果。这比字符串相等或正则匹配多了一步数学等价判断。

验证条件还要求生成以 `finish_reason=stop` 正常结束且最终 `content` 非空。Math-Verify 只检查最终数学答案，不判断中间推理是否正确。通过后，Teacher 的完整 response 进入 SFT 数据。

| 阶段 | 示例内容 |
| --- | --- |
| prompt | `12 个苹果平分给 3 人，每人几个？` |
| Teacher response | `12 / 3 = 4，所以每人 4 个。\boxed{4}` |
| gold answer | `4` |
| Math-Verify | 通过 |
| SFT 记录 | `user=prompt`，`assistant=Teacher response` |

In [2]:
# 用固定 seed 展示本阶段复用的 GSM8K row IDs。
sequence_config = load_sequence_teacher_config(
    CONFIG_DIR / "sequence_teacher.json"
)
split = build_gsm8k_split(
    7473,
    seed=sequence_config.seed,
    development_rows=sequence_config.development_rows,
    teaching_rows=sequence_config.teaching_rows,
)
print(
    f"reserved development: {len(split['development_row_ids']):,} "
    "(not used by this SFT)"
)
print(f"candidate training area: {len(split['training_row_ids']):,}")
print(f"Teacher prompts actually used: {len(split['teaching_100_row_ids']):,}")
print(f"first five row IDs: {split['teaching_100_row_ids'][:5]}")

reserved development: 500 (not used by this SFT)
candidate training area: 6,973
Teacher prompts actually used: 100
first five row IDs: [6591, 4437, 3510, 957, 1666]


这三个数字的用途不同：

```text
GSM8K 官方 train：7,473 条
├── reserved development：500 条
│   └── 为后续统一评估预留；本阶段没有用它计算 validation loss
└── candidate training area：6,973 条
    └── 本阶段实际发送给 Teacher：100 条
        └── Math-Verify 通过：87 条
            ├── Sequence SFT train：77 条
            └── Sequence SFT validation：10 条
```

本阶段实际发送给 Teacher 的是 100 条，实际用于 Student 训练的是 77 条；结果中的 Sequence-level validation loss 来自另外 10 条 Teacher response。500 条 reserved development 留给后续统一评估。

GSM8K 的 `answer` 同时包含官方解题过程和最终值。`####` 后的内容是自动验证使用的 gold answer。

In [3]:
# 将 GSM8K 完整 answer 拆出可验证的最终值。
example_gold = "12 / 3 = 4.\n#### 4"
example_response = "12 / 3 = 4, so the answer is \\boxed{4}."
print(f"complete answer: {example_gold!r}")
gold_value = gsm8k_gold_answer(example_gold)
print(f"value for verification: {gold_value!r}")
print(f"Teacher response: {example_response!r}")
print(f"Math-Verify passed: {verify_math_answer(gold_value, example_response)}")

complete answer: '12 / 3 = 4.\n#### 4'
value for verification: '4'
Teacher response: '12 / 3 = 4, so the answer is \\boxed{4}.'
Math-Verify passed: True


## 3. Logit Distillation 在哪些位置计算

Teacher 和 Student 接收完全相同的 `input_ids`。记 batch size 为 $B$，sequence length 为 $T$，vocabulary size 为 $V$。

| Tensor | Shape | 含义 |
| --- | --- | --- |
| `input_ids` | $[B, T]$ | Teacher 和 Student 共用的 token IDs |
| Student logits | $[B, T, V]$ | Student 对每个位置的词表打分 |
| Teacher logits | $[B, T, V]$ | Teacher 对同一位置的词表打分 |
| `labels` | $[B, T]$ | gold token ID 或 `-100` |

Causal shift 后，位置 $t$ 的 logits 预测 `labels[t + 1]`。`labels[t + 1] == -100` 时，该 logits 位置不进入 CE 和 KD。因此阶段 5 已经学过的 response mask 也决定 KD 的计算位置。

In [4]:
# 用小 Tensor 查看 causal shift 后的有效 KD 位置。
torch.manual_seed(7)
student_logits = torch.randn(1, 5, 6, requires_grad=True)
teacher_logits = torch.randn(1, 5, 6)
labels = torch.tensor([[-100, -100, -100, 4, 2]])
losses = distillation_losses(
    student_logits,
    teacher_logits,
    labels,
    ce_weight=0.5,
    temperature=1.5,
)
print(f"Student logits:   {tuple(student_logits.shape)}")
print(f"Teacher logits:   {tuple(teacher_logits.shape)}")
print(f"labels:           {tuple(labels.shape)}")
print(f"supervised tokens: {losses.supervised_tokens}")

Student logits:   (1, 5, 6)
Teacher logits:   (1, 5, 6)
labels:           (1, 5)
supervised tokens: 2


上例的 labels 是 `[-100, -100, -100, 4, 2]`。Causal shift 后的对应关系是：

| logits 位置 | 预测的 label | 进入 CE / KD |
| ---: | ---: | --- |
| 0 | `labels[1] = -100` | 否 |
| 1 | `labels[2] = -100` | 否 |
| 2 | `labels[3] = 4` | 是 |
| 3 | `labels[4] = 2` | 是 |

因此 CE 和 KD 汇总 logits 位置 2 和 3。这两个 response 预测位置依赖前面 prompt 的 Transformer 计算，Teacher forward 仍然接收整条 `input_ids`。

## 4. Temperature 改变概率分布

记第 $i$ 个 token 的 logit 为 $z_i$，temperature 为 $T$，缩放后的概率为 $p_i^{(T)}$：

$$p_i^{(T)} = \operatorname{softmax}(z_i / T)$$

$T = 1$ 保留原分布。$T > 1$ 缩小 logits 差距，概率更平缓。

In [5]:
# 比较同一组 Teacher logits 在两个 temperature 下的概率。
teacher_example = torch.tensor([4.0, 2.0, 1.0, 0.0])
print("token     T=1       T=2")
probability_t1 = torch.softmax(teacher_example, dim=-1)
probability_t2 = torch.softmax(teacher_example / 2.0, dim=-1)
for token, p1, p2 in zip(["A", "B", "C", "D"], probability_t1, probability_t2):
    print(f"{token:<5} {p1.item():>8.4f} {p2.item():>9.4f}")

token     T=1       T=2
A       0.8310    0.5793
B       0.1125    0.2131
C       0.0414    0.1293
D       0.0152    0.0784


$T=2$ 时，最高概率 token A 的占比下降，B、C、D 的概率上升。Student 因此能看到 Teacher 对其他 token 的相对偏好。

对 Forward KL 求 Student logit $z_{S,k}$ 的 gradient，可得：

$$\frac{\partial D_{KL}(p_T^{(T)}\Vert p_S^{(T)})}{\partial z_{S,k}}=\frac{p_{S,k}^{(T)}-p_{T,k}^{(T)}}{T}$$

式子外面有一个 $1/T$。temperature 较高时，Teacher 与 Student 的概率差 $p_S^{(T)}-p_T^{(T)}$ 通常也约按 $1/T$ 缩小，因此 gradient 量级约按 $1/T^2$ 缩小。这不是所有 gradient 统一乘同一个常数：temperature 已经改变概率分布，每个 token 的概率差会分别变化。

本阶段将 KD loss 乘以 $T^2$，用来补偿 temperature 对 gradient 量级的附带缩放。temperature 主要控制分布平滑程度，`ce_weight` 再单独控制 CE 与 KD 的比例。

## 5. CE、KD 与混合 loss

以 gold next token 为 `4` 的一个位置为例：

- CE 只读取 gold token ID，计算 $L_{CE}=-\log p_S(4)$。其他 token 没有各自的 Teacher 概率；
- KD 读取 Teacher 在完整 vocabulary 上的分布，例如 `4: 0.82`、`four: 0.10`、`3: 0.03`，再用 KL divergence 与 Student 分布比较；
- `ce_weight` 决定两种信号的混合比例。

$$L = \alpha L_{CE} + (1-\alpha)T^2D_{KL}(p_T^{(T)} \Vert p_S^{(T)})$$

$\alpha$ 是配置中的 `ce_weight`。这里使用 Forward KL：Teacher 分布放在前面，Teacher 概率较高的 token 都会推动 Student 分配概率。gradient 经由 Student 分布回传。

训练代码按 weight 跳过不参与目标的分支：

| `ce_weight` | 当前 step 计算什么 |
| ---: | --- |
| `1` | 只计算 CE；跳过 Teacher forward 与 KD |
| `0` | 只计算 Teacher forward 与 KD；跳过 CE |
| `0 < ce_weight < 1` | 同时计算 CE 与 KD |

validation 仍统一计算 CE、Forward KL 和 Reverse KL，使不同训练目标保留相同的比较口径。

### Forward KL 与 Reverse KL

两种 KL 使用相同的 Teacher 和 Student 概率，只交换两个分布的位置：

$$D_{KL}(p_T \Vert p_S)=\sum_i p_T(i)\log\frac{p_T(i)}{p_S(i)}$$

$$D_{KL}(p_S \Vert p_T)=\sum_i p_S(i)\log\frac{p_S(i)}{p_T(i)}$$

假设 Teacher 对三个 token 的概率是 `[0.80, 0.15, 0.05]`，Student 是 `[0.45, 0.45, 0.10]`：

- Forward KL 按 Teacher 概率加权。Teacher 认为可能的 token 都会影响 loss，Student 倾向于覆盖 Teacher 的概率分布；
- Reverse KL 按 Student 概率加权。Student 会重点压低 Teacher 概率很小而自己概率较高的 token，概率更容易集中到 Teacher 的高概率区域。

`kl_direction` 决定纯 KD 训练采用哪个方向。两种方向都使用 response mask 和 $T^2$。

In [6]:
# 保持 logits 不变，只切换 CE 与 KD 的权重。
print("setting       CE        KD     total")
settings = [
    ("CE", 1.0, "forward"),
    ("Forward KD", 0.0, "forward"),
    ("Reverse KD", 0.0, "reverse"),
    ("CE + KD", 0.5, "forward"),
]
for name, ce_weight, kl_direction in settings:
    current = distillation_losses(
        student_logits,
        teacher_logits,
        labels,
        ce_weight=ce_weight,
        temperature=1.5,
        kl_direction=kl_direction,
    )
    ce_text = "-" if current.ce is None else f"{current.ce.item():.4f}"
    kd_text = "-" if current.kd is None else f"{current.kd.item():.4f}"
    print(f"{name:<10} {ce_text:>8} {kd_text:>9} {current.total.item():>9.4f}")

setting       CE        KD     total
CE           2.1928         -    2.1928
Forward KD        -    0.5813    0.5813
Reverse KD        -    0.6595    0.6595
CE + KD      2.1928    0.5813    1.3870


## 6. Backward 更新哪个模型

Logit Distillation 的训练图可以写成：

```text
input_ids ─┬→ Teacher forward → teacher logits ─────────────┐
           └→ Student forward → student logits ─┬→ KD loss ─├→ total loss
labels ─────────────────────────────└→ CE loss ─┘           │
                                                            └→ backward → Student
```

Teacher 的作用是生成目标分布。`eval()` 固定 dropout 等运行行为，`no_grad()` 停止记录 Teacher 计算图。Backward 沿 Student logits 回传，optimizer 只更新 Student parameters。

Teacher weights、当前 forward 的中间 Tensor 和输出 logits 仍占用显存。这些中间 Tensor 使用完即可释放；`no_grad()` 还会省去 Teacher backward 所需的计算图和 parameter gradients。

### Teacher logits 为什么不保存

本阶段采用在线计算。每个训练 batch 依次完成：

```text
读取一组 input_ids
  → Teacher forward，得到当前 batch 的 teacher logits
  → Student forward，得到 student logits
  → 计算 KD，backward 只更新 Student
  → 释放当前 batch 的 teacher logits
```

训练只有一个 epoch，4,096 条训练样本各经过一次，因此同一训练位置不会因多个 epoch 被重复计算。固定 validation 会在每次评估时重新执行 Teacher forward。若训练多个 epoch，预计算 logits 可以减少重复 Teacher forward，同时需要额外的磁盘写入、存储和读取。

## 7. 本阶段的五组对照

Student 使用阶段 5 自训练 Full SFT 权重，Teacher 使用官方 MiniMind 198M MoE Full SFT 权重。五组配置共用这两个起点、train row IDs、validation row IDs、batch size、optimizer steps 和 learning rate。

In [7]:
# 读取五组配置，核对实际改变的 loss 参数。
print("profile              CE     KD   direction      T   steps")
filenames = ["ce.json", "kd.json", "reverse_kd.json", "ce_kd.json", "ce_kd_t2.json"]
for filename in filenames:
    config = load_logit_distillation_config(CONFIG_DIR / filename)
    steps = (config.train_rows // config.effective_batch_size) * config.epochs
    print(
        f"{config.profile:<19} {config.ce_weight:>3.1f} "
        f"{1 - config.ce_weight:>6.1f} {config.kl_direction:>11} "
        f"{config.temperature:>6.1f} {steps:>7}"
    )

profile              CE     KD   direction      T   steps
distill_ce          1.0    0.0     forward    1.5     256
distill_kd          0.0    1.0     forward    1.5     256
distill_reverse_kd  0.0    1.0     reverse    1.5     256
distill_ce_kd       0.5    0.5     forward    1.5     256
distill_ce_kd_t2    0.5    0.5     forward    2.0     256


`kd.json` 与 `reverse_kd.json` 只改变 KL direction；`ce_kd.json` 与 `ce_kd_t2.json` 只改变 temperature。validation 为每组统一记录 CE、Forward KL 和 Reverse KL，因此比较 KL 方向时不需要混用不同含义的 `kd_loss`。

## 8. 执行顺序

命令和产物路径集中记录在 `docs/stages/08_distillation/README.md`。实际执行顺序是：

1. 在 LM Studio 加载本地 Qwen3.5-9B GGUF 并开启 Local Server；
2. 生成 GSM8K 固定划分；
3. 生成 100 条 Teacher responses 并用 Math-Verify 验证；
4. 用通过验证的 response 运行短 SFT；
5. 下载 MiniMind MoE Teacher；
6. 运行五组 CE / KD 对照；
7. 把生成通过率、validation CE、Forward KL、Reverse KL、显存和用时写入 `RESULTS.md`。

## 9. 实际实验结果

### 9.1 Sequence-level 数据与训练结果

100 条 Teacher prompts 中，90 条正常结束，10 条达到生成长度上限。正常结束的结果再经过 Math-Verify，最终有 87 条进入 SFT 数据。它们被划分为 77 条 train 和 10 条 validation。这里的 validation loss 只衡量 Student 对 Teacher response tokens 的预测，不是 reserved development 上的 GSM8K 答题准确率。

### 9.2 Logit Distillation 应该比较哪几个数

每组先看自己的训练目标是否从初始值下降。训练目标中的 KD 部分是 scaled KD loss：

$$L_{KD}=T^2\times D_{KL}$$

再用统一 validation 口径查看 CE、raw Forward KL 和 raw Reverse KL。raw KL 不含 $T^2$；temperature 不同时，softmax 分布也不同，因此 temperature 对照主要看各组相对自身起点的变化。

训练产物统一使用以下字段：`forward_kl` 和 `reverse_kl` 记录 raw KL；`forward_kd_loss` 和 `reverse_kd_loss` 记录乘过 $T^2$ 的 scaled KD loss。

In [8]:
import json

sequence_summary = json.loads(
    (ROOT / "outputs/minimind/stage8/sequence_sft/summary.json").read_text()
)
sequence_metrics = [
    json.loads(line)
    for line in (ROOT / "outputs/minimind/stage8/sequence_sft/metrics.jsonl").read_text().splitlines()
    if line
]
print("Sequence-level Distillation")
print("  verified:       87/100 (87%)")
print(f"  validation CE:  {sequence_metrics[0]['loss']:.4f} -> {sequence_summary['final_evaluation']['loss']:.4f}")
print(f"  wall time:      {sequence_summary['wall_time_seconds']:.1f} s")

profiles = [
    ("CE", "ce"),
    ("Forward KD", "kd"),
    ("Reverse KD", "reverse_kd"),
    ("CE + Forward KD", "ce_kd"),
    ("CE + Forward KD, T=2", "ce_kd_t2"),
]
summaries = {}
metric_histories = {}
for name, directory in profiles:
    summaries[name] = json.loads(
        (ROOT / f"outputs/minimind/stage8/logit/{directory}/summary.json").read_text()
    )
    metric_histories[name] = [
        json.loads(line)
        for line in (ROOT / f"outputs/minimind/stage8/logit/{directory}/metrics.jsonl").read_text().splitlines()
        if line
    ]

print("\nTraining objective")
print("profile                      T    initial      final")
for name, _ in profiles:
    summary = summaries[name]
    initial_loss = metric_histories[name][0]["loss"]
    final_loss = summary["final_evaluation"]["loss"]
    print(f"{name:<27} {summary['temperature']:>3.1f} {initial_loss:>10.4f} {final_loss:>10.4f}")

print("\nFinal validation metrics")
print("profile                     CE   raw F-KL   raw R-KL   memory      time")
for name, _ in profiles:
    summary = summaries[name]
    result = summary["final_evaluation"]
    memory_gib = summary["peak_memory_bytes"] / 1024**3
    print(
        f"{name:<23} {result['ce_loss']:>7.4f} {result['forward_kl']:>10.4f} "
        f"{result['reverse_kl']:>10.4f} {memory_gib:>6.2f} GiB "
        f"{summary['wall_time_seconds']:>7.1f} s"
    )

Sequence-level Distillation
  verified:       87/100 (87%)
  validation CE:  1.0869 -> 0.8498
  wall time:      50.2 s

Training objective
profile                      T    initial      final
CE                          1.5     1.5974     1.5963
Forward KD                  1.5     0.8934     0.8537
Reverse KD                  1.5     1.0890     0.9262
CE + Forward KD             1.5     1.2454     1.2318
CE + Forward KD, T=2        2.0     1.5814     1.5268

Final validation metrics
profile                     CE   raw F-KL   raw R-KL   memory      time
CE                       1.5963     0.3970     0.4843   2.55 GiB    78.3 s
Forward KD               1.6079     0.3794     0.4401   2.73 GiB   214.8 s
Reverse KD               1.6333     0.3963     0.4117   2.73 GiB   124.3 s
CE + Forward KD          1.6001     0.3838     0.4526   2.79 GiB   126.7 s
CE + Forward KD, T=2     1.6080     0.3614     0.4002   2.79 GiB   127.1 s


## 10. 用指标验收本次实验

### 10.1 Sequence-level 链路

| 指标 | 本次结果 | 判断 |
| --- | --- | --- |
| Teacher 完整生成 | 90 / 100 | 10 条达到长度上限，没有产生可用最终回答 |
| Math-Verify 自动通过 | 87 / 100 | 87 条直接进入 SFT 数据 |
| 3 条完整但未通过 | 2 条答案与 gold 不一致；1 条给出 240 分钟，但 boxed answer 写成 4 小时 | 自动通过率会把单位等价但数值形式不同的回答判为失败 |
| Sequence validation loss | 1.0869 → 0.8498，下降 21.8% | Student 学会了这批 Teacher response 的 token 分布 |

这条链路完成了“Teacher 生成 → 自动验证 → Student SFT”。10 条 validation response 足以检查训练方向，无法替代 500 条 reserved development 上的数学答题评估。

### 10.2 Logit Distillation

所有配置从相同 Student checkpoint 开始。`T=1.5` 组的共同初始 CE 为 1.5974，raw Forward KL 为 0.3971，raw Reverse KL 为 0.4840。

| 配置 | 目标指标变化 | validation CE 变化 | 判断 |
| --- | --- | --- | --- |
| CE | CE 1.5974 → 1.5963 | 下降 0.0011 | 256 steps 对已经完成 SFT 的 Student 改动很小 |
| Forward KD | raw Forward KL 0.3971 → 0.3794 | 1.5974 → 1.6079 | 更接近 Teacher 分布，同时 gold-token CE 略有上升 |
| Reverse KD | raw Reverse KL 0.4840 → 0.4117 | 1.5974 → 1.6333 | Reverse KL 明显下降，gold-token CE 上升最多 |
| CE + Forward KD | raw Forward KL 0.3971 → 0.3838 | 1.5974 → 1.6001 | 在保持 CE 和贴近 Teacher 之间取得折中 |
| CE + Forward KD, T=2 | 本组 scaled objective 1.5814 → 1.5268 | 1.5974 → 1.6080 | 更平缓的分布得到有效优化；与本组起点比较，不用 scaled loss 横比 `T=1.5` |

### 10.3 资源结果

CE baseline 使用 2.55 GiB；纯 KD 使用约 2.73 GiB；混合 CE + KD 使用约 2.79 GiB。额外显存来自 Teacher weights、Teacher forward 和 KD 中间量。Reverse KD 与混合组约用 124～127 秒，CE baseline 用 78.3 秒，说明在线 Teacher forward 增加了每 step 成本。Forward KD 的 214.8 秒明显偏离相同计算规模的其他运行，只作为本次 wall time 记录。

### 10.4 本阶段结论

两条蒸馏数据流均已走通，五组训练目标也都按预期下降。当前实验支持三个判断：Sequence-level 可以跨 tokenizer 传递 Teacher 文本；Forward 与 Reverse KL 会产生不同的 CE–KL 取舍；Logit Distillation 通过在线 Teacher forward 换取完整词表分布信号。模型数学能力是否改善，还需要在 reserved development 上执行生成评估。

## 11. 下一阶段前瞻

阶段 9 将暂停增加训练算法，统一整理配置、运行记录、checkpoint 恢复和评估口径。阶段 8 的固定 split、五组配置和结果表会成为该阶段的直接输入。